# Notebook 03: Exploratory Data Analysis (EDA)
## Univariate, Bivariate, Regional & Temporal Patterns across Formula 1 (2010–2019)

### Analytical Objectives:
1. Examine univariate distributions, central tendencies, and dispersions across sporting and commercial metrics.
2. Analyze longitudinal time-series trends across 10 championship seasons.
3. Compare regional attendance differences across Continents (Europe, Asia, Americas, Middle East, Oceania).
4. Benchmark Buddh International Circuit against global baseline quartiles and peer inaugural circuits.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
f1_df = pd.read_csv("../data/processed/f1_races_clean.csv")
ind_df = pd.read_csv("../data/processed/india_gp_clean.csv")
print(f"Loaded {len(f1_df)} baseline races and {len(ind_df)} Indian GP editions.")

Loaded 198 baseline races and 3 Indian GP editions.


### 1. Descriptive Summary Statistics of Baseline Numerical Metrics

In [2]:
metrics = [
    'attendance_race_day', 'attendance_weekend', 'average_speed_kmh',
    'winning_margin_s', 'finishers_count', 'dnf_count', 'total_pit_stops',
    'circuit_length_km', 'gdp_per_capita_usd', 'event_tenure_years'
]

summary_stats = f1_df[metrics].describe().T
summary_stats['IQR'] = summary_stats['75%'] - summary_stats['25%']
summary_stats['Skewness'] = f1_df[metrics].skew()
summary_stats['Kurtosis'] = f1_df[metrics].kurtosis()

pd.set_option('display.float_format', lambda x: '%.2f' % x)
summary_stats[['mean', 'std', 'min', '50%', 'max', 'IQR', 'Skewness', 'Kurtosis']]

,mean,std,min,50%,max,IQR,Skewness,Kurtosis
attendance_race_day,77439.39,27090.82,28000.00,74500.00,141000.00,35000.00,0.40,-0.38
attendance_weekend,194562.63,84313.87,57600.00,177300.00,366600.00,129925.00,0.48,-0.87
average_speed_kmh,190.68,26.89,74.90,194.80,246.60,22.55,-1.15,2.59
winning_margin_s,7.92,7.43,0.30,5.00,37.80,8.40,1.43,1.83
finishers_count,16.95,2.08,11.00,17.00,24.00,2.00,-0.11,0.66
dnf_count,4.72,2.10,0.00,5.00,11.00,3.00,0.22,-0.27
total_pit_stops,37.11,14.73,16.00,34.00,84.00,18.00,1.11,0.84
circuit_length_km,5.19,0.80,3.34,5.41,7.00,1.41,-0.17,0.26
gdp_per_capita_usd,39734.82,35828.81,1443.00,38775.00,195000.00,34312.25,2.62,8.05
event_tenure_years,30.76,22.98,1.00,26.50,70.00,39.75,0.36,-1.30


### 2. Longitudinal Season-over-Season Attendance Trends (2010–2019)

In [3]:
season_trends = f1_df.groupby('season').agg(
    races=('race_id', 'count'),
    mean_sunday=('attendance_race_day', 'mean'),
    median_sunday=('attendance_race_day', 'median'),
    mean_weekend=('attendance_weekend', 'mean')
).reset_index()

print("Longitudinal Attendance Trends:")
print(season_trends)

Longitudinal Attendance Trends:
   season  races  mean_sunday  median_sunday  mean_weekend
0    2010     19     72368.42       72000.00     177189.47
1    2011     19     73736.84       72000.00     175147.37
2    2012     20     75450.00       72500.00     185435.00
3    2013     19     72947.37       72000.00     179968.42
4    2014     19     76052.63       77000.00     192878.95
5    2015     19     79421.05       72000.00     203836.84
6    2016     21     74857.14       72000.00     190842.86
7    2017     20     80350.00       74000.00     206075.00
8    2018     21     83190.48       81000.00     212814.29
9    2019     21     84857.14       85000.00     217380.95


### 3. Regional Breakdown (Continental Comparison)

In [4]:
regional_summary = f1_df.groupby('continent').agg(
    races=('race_id', 'count'),
    mean_sunday=('attendance_race_day', 'mean'),
    median_sunday=('attendance_race_day', 'median'),
    std_sunday=('attendance_race_day', 'std'),
    mean_gdp=('gdp_per_capita_usd', 'mean')
).reset_index().sort_values(by='mean_sunday', ascending=False)

print("Regional Summary Statistics:")
print(regional_summary)

Regional Summary Statistics:
     continent  races  mean_sunday  median_sunday  std_sunday  mean_gdp
4      Oceania     10    103000.00      101500.00     6565.91  58523.70
0     Americas     33    102060.61      107000.00    23784.11  33157.58
2       Europe     85     79235.29       75000.00    24134.86  50317.49
1         Asia     51     65941.18       63000.00    19658.50  25307.88
3  Middle East     19     44052.63       50000.00    13133.81  32650.95


### 4. Indian Grand Prix Benchmarking vs Global Baseline

In [5]:
global_p25 = f1_df['attendance_race_day'].quantile(0.25)
global_p50 = f1_df['attendance_race_day'].quantile(0.50)
global_p75 = f1_df['attendance_race_day'].quantile(0.75)

print(f"Global Sunday Attendance Quartiles: 25th={global_p25:,.0f} | 50th={global_p50:,.0f} | 75th={global_p75:,.0f}\n")

for _, r in ind_df.iterrows():
    att = r['attendance_race_day']
    pctile = (f1_df['attendance_race_day'] < att).mean() * 100
    print(f"India GP {r['season']}: Sunday Attendance = {att:,} -> Baseline Percentile = {pctile:.1f}th")

Global Sunday Attendance Quartiles: 25th=60,000 | 50th=74,500 | 75th=95,000

India GP 2011: Sunday Attendance = 95,000 -> Baseline Percentile = 73.7th
India GP 2012: Sunday Attendance = 65,000 -> Baseline Percentile = 35.4th
India GP 2013: Sunday Attendance = 60,000 -> Baseline Percentile = 22.7th
